# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [32]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


Note: you may need to restart the kernel to use updated packages.


### Read parameters

In [33]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [34]:
# load parameters from CSV into dict
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

### Global variables

In [ ]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

# variables for hourly fairness and average weekly work hours
AVG_WEEKLY_HOURS    = float(params["avg_weekly_work_hours"])   # target avg weekly work hours
AVG_REFERENCE_WEEKS = int(params["avg_reference_weeks"])       # weeks window for average
MAX_WEEKLY_HOURS    = AVG_WEEKLY_HOURS * AVG_REFERENCE_WEEKS   # total hours over reference window
W_WORKERS  = int(params["obj_w_workers"])   # weight: minimize active weeks
W_FAIRNESS = int(params["obj_w_fairness"])  # weight: Pesch1 deviation from Soll
W_PESCH2 = int(params["obj_w_pesch2"])  # weight: Pesch2 equal shift-class proportion across cycles
W_CHANGEofCLASSES = int(params["obj_w_minChangeOfClasses"] # weight: objective to minimize change of shift classes 

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
MAX_NB_CYLCEs = int(params["max_nb_cycles"]) # number of cycles

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}



In [36]:
# basic inputs and parameters
cycles = range(1, MAX_NB_CYLCEs+1)
cycleWeeks  = range(1, MAX_CYCLE_WEEKS+1)
Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday (in line with static variable DICT_WEEKDAYS)

# improvements outstanding:
    # use files for parameter input
        # shift definitions => done
        # available staff => not required
        # user objectives: weighted priorities

### Shift Objects
creating shift objects:

In [37]:

# read input data for shift definitions
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")

shift_objects = abd.build_shift_objects(data_shiftSet)

# distuingish work shift from all shifts: 
#   WorkShifts are all shifts imported from the shift set file
#   other shifts are created hard-coded below (freeDay, reserveShift)
WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)

# additional hard-coded shifts for stand-bys and free days
freeDayShift = Shift.Shift("[freeDay_:-)_]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           0, 
                           1, 
                           0, 
                           False, 
                           None 
                           )
shift_objects.append(freeDayShift)

# optional enhancement: 
# use user-input for number of stand-bys 
# OR deviate need from the number of shifts to be staffed on a specific day (eg 5%)
reserveShift = Shift.Shift("[reserveShift]", 
                           "dummy for jump shifts", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           1, 
                           5, 
                           0, 
                           True, 
                           None 
                           )
shift_objects.append(reserveShift)

#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)




In [38]:
# pre-work for condition to have as less shift class changes as possible (target: same shift-class in blocks as far as possible)

# Map each shift id to its shift_class (eg 'dayShift': 1)
dict_shift_to_class = {s.shift_id: s.shift_class for s in shift_objects}

# unique list of shift classes from shift definition (from input file)
set_classes = sorted(set(dict_shift_to_class.values()))


In [39]:

# # preparation for output
# shift_info = {
#     s.shift_id: {
#         "start": s.start.strftime("%H:%M"),
#         "end": s.end.strftime("%H:%M"),
#         "workingTime": s.shift_work_time_assignment
#     }
#     for s in shift_objects
# }

In [40]:
# Pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days.
# Excludes dummy shifts (freeDay, spareShift) since they have no real start/end times.
# Result: list of (sh1_id, sh2_id) tuples that cannot appear on consecutive days in a snake.
DUMMY_SHIFTS = {"[freeDay_:-)_]", "[reserveShift]"}
incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]

In [41]:
# Pre-compute work hours per shift using shift_duration_hours from Shift.py.
# 
# freeDay and spareShift are excluded - they have no real duration.
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    if s.shift_id not in DUMMY_SHIFTS
}

# net work time per shift (break excluded); fall back to presence time if not specified ('none')
work_hours = {
    s.shift_id: (float(s.shift_work_time_assignment)
                 if s.shift_work_time_assignment is not None
                 else Shift.shift_duration_hours(s))
    for s in shift_objects
    if s.shift_id not in DUMMY_SHIFTS
}

#when none then gleich wie shift hours 

### modelling

$x \to$ x  
$y \to$ active\_week  
$z \to$ active\_cycle  

In [42]:
# modelling

modelCycle = gp.Model("SnakeBuilding_simple")

# --- decision variables
# x[c, s, d, sh] = 1 if in cycle c, cycleWeek s, on weekday d, shift sh is assigned
x = modelCycle.addVars(cycles, cycleWeeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all, 0 otherwise
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_week[c,s] = 1 if cycle c uses cycleWeek s (i.e., at least one shift in that week is active), 0 otherwise
active_week = modelCycle.addVars(cycles, cycleWeeks, vtype=GRB.BINARY, name="active_week")


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{c, s,d,w}} >= 1$$
$$    \forall d \in D,$$
$$ \forall w \in N,$$
$$ \forall c \in M$$

$x_{c,s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [43]:
# basic constraints

# ensure a cycleWeek can only be active when the according cylce is active
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(active_week[c,w] - active_cycle[c] <= 0,
                             name=f"WeekImpliesCycle_c{c}_s{w}")

# enforce active_week >= any assignment in that week
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, sh] for d in Weekdays for sh in Shifts) <= len(Weekdays) * len(Shifts) * active_week[c, w],
            name=f"Link_x_activeWeek_c{c}_s{w}_upper"
        )

# active_cycle must be 1 if any week in that cycle is active
# "no active cycleWeek without active cycle"
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) >= active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}")


#### condition c04  

ensure that cycles and cycleWeeks are activated in ascending order  

$y_s - y_{s+1} >= 0, \forall s \in \{1,\dots,M-1\}, \forall c \in C$  
$z_c - z_{c+1} >= 0, \forall c \in \{1,\dots,C-1\}$

In [44]:

for c in range(1, MAX_NB_CYLCEs): # loop from first to second-last entry
    modelCycle.addConstr(active_cycle[c] - active_cycle[c + 1] >= 0,
                         name=f"CycleOrder_c{c}")

for c in cycles: # loop across all possible cycles
    for w in range(1, MAX_CYCLE_WEEKS): # loop from first to second-last entry
        modelCycle.addConstr(active_week[c, w] - active_week[c, w + 1] >= 0,
                             name=f"WeekOrder_c{c}_s{w}")


In [45]:
for d in Weekdays: #loop over all week days
    for ws in WorkShifts: # loop over all work shift
        shift = next(s for s in shift_objects if s.shift_id == ws) # next is an alternative for 'for s in shift_objects: if s.shift_id == ws: shift = s'
#       if DICT_WEEKDAYS[shift.weekdays[0]] <= d <= DICT_WEEKDAYS[shift.weekdays[-1]]:
        if d in [DICT_WEEKDAYS[w] for w in shift.weekdays]: # correction to also cover shifts that appear for non-consecutive week days (eg. Mon, Wed)
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) >= 1,
                        name=f"Cover_day{d}_{ws}")
        else:
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) <= 0,
                        name=f"Cover_day{d}_{ws}")

#Логика: перед добавлением ограничения проверяем входит ли день d в список weekdays этой смены. Если нет ограничение не добавляется, смена в этот день не требуется.
#Logic: Before adding a restriction, we check whether day d is included in the list of weekdays for this shift. If not, the restriction is not added, as no shift is required on that day.

# 1. each shift has to be covered on each day
#for d in Weekdays:
#    for ws in WorkShifts:
#        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
#                    name=f"Cover_day{d}_{ws}")



#### condition c02

<div align="left">

$ \sum_{s \in S}{x_{c,w,d,s}} - y_s = 0$  
$ \forall c \in C, \forall w \in W, \forall d \in D $

</div>

In [46]:
# 2. each cycle shall have exactly one shift per day (implying that on cycleWeek has exactly one shift per day)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, w, d, sh] for sh in Shifts) - active_week[c, w] == 0,
                name=f"OneShiftPerDay_c{c}_s{w}_d{d}"
            )


#### condition c03

$ \sum_{d = 1}^{8-MAX\_CONSEC\_DAYS+1}{x_{}} $  
$ \forall c \in C, \forall w \in W, \forall d \in {1,\dots, 8 - MAX\_CONSEC\_DAYS +1} $

In [47]:
### UPDATED // DOUBLE-CHECK!!!
### => is there a minimum rest time after 5 days working???
### => or should this condition rather be "max 5 working days within 7 days"???

# # 3) at max 5 consecutive working days (ensure time for resting)
# for c in cycles:
#     for w in cycleWeeks:
#         for start in range(1, 8 - MAX_CONSEC_DAYS + 1):
#             modelCycle.addConstr(
#                 gp.quicksum(x[c, w, d, sh] for d in range(start, start + MAX_CONSEC_DAYS) for sh in WorkShifts)
#                 <= MAX_CONSEC_DAYS,
#                 name=f"Max{MAX_CONSEC_DAYS}Work_c{c}_s{w}_start{start}"
#             )


# Helper function: convert a global day index t into tupel (cycleWeek, weekday)
# This allows to treat all days across all cycleWeeks as one continuous timeline.
def decode_global_day(t):
    s = (t - 1) // 7 + 1 # Compute cycleWeek index from global day t (1-based indexing)
    d = (t - 1) % 7 + 1 # Compute weekday index from global day t (1-based indexing)
    return s, d

# total number of days across all cycleWeeks
# used to define the global timeline over which consecutive work days are checked.
TOTAL_DAYS = MAX_CYCLE_WEEKS * 7

# limit the number of consecutive working days across all cycleWeeks
# for each cycle scan through the entire global timeline and check every (MAX_CONSEC_DAYS + 1)-days window
# to ensure that not all days in the window are working days.
for c in cycles:

    # Iterate over all possible start positions of a sliding window
    # The window length is MAX_CONSEC_DAYS + 1, so the last valid start is:
    # TOTAL_DAYS - MAX_CONSEC_DAYS
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_DAYS + 1):

        # Collect expressions representing work indicators for each day in the window
        moving_time_window = []

        # Iterate through each offset inside the window
        # offset = 0 means the first day of the window
        # offset = MAX_CONSEC_DAYS means the last day of the window
        for offset in range(0, MAX_CONSEC_DAYS + 1):

            # Compute the global day index inside the window
            t = t_start + offset

            # Convert global day index back to (cycleWeek, weekday)
            s, d = decode_global_day(t)

            # work[c,s,d] = sum of all work shifts assigned on that day
            # If any WorkShift is assigned, this sum becomes 1 (binary model)
            moving_time_window.append(
                gp.quicksum(x[c, s, d, sh] for sh in WorkShifts)
            )

        # ACTUAL CONSTRAINT COMES HERE:
        # In any time window of length MAX_CONSEC_DAYS + 1,
        # the number of working days must be <= MAX_CONSEC_DAYS.
        # This prevents sequences of MAX_CONSEC_DAYS + 1 consecutive working days.
        modelCycle.addConstr(
            gp.quicksum(moving_time_window) <= MAX_CONSEC_DAYS,
            name=f"MaxConsecDays_c{c}_t{t_start}"
        )


### QUESTION: How many days rest after 5 days working?

#### Condition c05

In [48]:
# c05: minimum rest time between consecutive shifts within a snake week.
# If sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same snake.
# Covers days 1-6 only; wrap-around (day 7 -> day 1) not yet modelled. => DONE NOW
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:  # for Mon to Sat use pairs of day d and d+1
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w, d+1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            elif d == 7 and w < MAX_CYCLE_WEEKS:  # Sun of week w -> Mon of NEXT week w+1 (snake runs continuously)
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w+1, 1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            # FIX (C05 wrap-around): the Sun->Mon transition below originally paired
            # Sunday of week w with Monday of the SAME week w. In a snake the weeks run
            # continuously, so Sunday of week w is followed by Monday of the NEXT week (w+1).
            # The old version constrained a non-existent backward pair and left the real
            # week-to-week rest gap unchecked. Now paired with w+1, guarded by w < MAX_CYCLE_WEEKS
            # (the last week has no successor). Old code kept commented below for reference.

#############
#            elif d == 7: # ensure this condition also for move from Sun to Mon (pair of days: d and 1)
#                for (sh1, sh2) in incompatible_pairs:
#                    modelCycle.addConstr(
#                        x[c, w, d, sh1] + x[c, w, 1, sh2] <= 1,
#                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
#                    )

#### Condition c06

In [49]:
# c06: total work hours per active snake week must not exceed MAX_WEEKLY_HOURS.
# Ensures no snake week accumulates more than the allowed weekly work time.
# Bound scales with active[s] so inactive snakes are not constrained.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(
               #shift_hours.get(sh, 0) * x[c, w, d, sh] # replaced by more efficient version without "get()":
                work_hours[sh] * x[c, w, d, sh]
                for d in Weekdays
               #for sh in Shifts if sh in shift_hours # # replaced by more efficient version:
                for sh in work_hours.keys()
            ) <= MAX_WEEKLY_HOURS * active_week[c, w],
            name=f"MaxWeeklyHours_c{c}_w{w}"
        )

#### Condition c07  
ensure balanced [almost equal] lenght of cycles (same number of cycleWeeks plus/minus 1 week)

$ W_c = \sum_{s=1}^{M} y_{c,s} \text{  : counting active weeks within cycle } c$  

$M: \text{MAX\_CYCLE\_WEEKS}$  

$\text{condition: }|W_c - W_{c'}| \le 1 \quad \forall c,c' \in C$  

$\text{linearized condition: }$  
$W_c - W_{c'} \le 1 \quad \forall c,c' \in C$  
$W_{c'} - W_c \le 1 \quad \forall c,c' \in C$

In [50]:
# c07: ensure all used cycles have balanced number of cycleWeeks;
# deviations of at most 1 week are allowed

# determine number of active weeks per cycle
# (W_c in formula)
cycle_length = {c: gp.quicksum(active_week[c, s] for s in cycleWeeks) for c in cycles}

# pairwise balance constraints
# |W_c1 - W_c2| <= 1  for all cycles c1 != c2 => implemented as 2 linear statements
for c1 in cycles:
    for c2 in cycles:
        if c1 < c2:  # avoiding duplicates and self-pairing
            # W_c1 - W_c2 <= 1
            modelCycle.addConstr(
                cycle_length[c1] - cycle_length[c2] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c2]), name=f"CycleBalance_upper_c{c1}_c{c2}") ## upper bound: cycle c1 cannot be more than 1 week longer than c2 — relaxed if c2 is inactive
            # W_c2 - W_c1 <= 1
            modelCycle.addConstr(cycle_length[c2] - cycle_length[c1] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c1]), name=f"CycleBalance_lower_c{c1}_c{c2}") ## lower bound: cycle c2 cannot be more than 1 week longer than c1 — relaxed if c1 is inactive
# Big-M: if cycle c is inactive (active_cycle=0), the right side becomes 1+MAX_CYCLE_WEEKS, which is always larger than any possible difference — so the constraint has no effect

In [51]:
# Pesch1: deviation of average weekly net work time from target (AVG_WEEKLY_HOURS)
#cycle_hours = {
#    c: gp.quicksum(work_hours[sh] * x[c, w, d, sh]
#                   for w in cycleWeeks for d in Weekdays for sh in work_hours.keys())
#    for c in cycles
#}
#dev_pos = modelCycle.addVars(cycles, lb=0, name="dev_pos")
#dev_neg = modelCycle.addVars(cycles, lb=0, name="dev_neg")
#
#for c in cycles:
#    modelCycle.addConstr(
#        cycle_hours[c] - AVG_WEEKLY_HOURS * cycle_length[c] == dev_pos[c] - dev_neg[c],
#        name=f"Pesch1_dev_c{c}"
#    )

#####
# Pesch1: per-week deviation of net work time from the weekly target (AVG_WEEKLY_HOURS).
# Soft target: the objective penalises deviation, it does NOT force exactly 40h.
# Measured per week so heavy/light weeks cannot cancel out across the cycle.
week_hours = {
    (c, w): gp.quicksum(work_hours[sh] * x[c, w, d, sh]
                        for d in Weekdays for sh in work_hours.keys())
    for c in cycles for w in cycleWeeks
}

dev_pos = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_pos")
dev_neg = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_neg")

for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            week_hours[(c, w)] - AVG_WEEKLY_HOURS * active_week[c, w] == dev_pos[c, w] - dev_neg[c, w],
            name=f"Pesch1_dev_c{c}_w{w}"
        )



In [52]:
# Pesch2: equal proportion of shift categories (by presence time) across cycles.
# Category = shift_class value. Uses shift_hours (presence / Anwesenheitszeit), NOT work_hours.
shift_class_of = {s.shift_id: s.shift_class for s in shift_objects if s.shift_id not in DUMMY_SHIFTS}
categories = sorted(set(shift_class_of.values()))

# presence hours of each category k inside each cycle c
cat_hours = {
    (c, k): gp.quicksum(shift_hours[sh] * x[c, w, d, sh]
                        for w in cycleWeeks for d in Weekdays
                        for sh in shift_hours.keys() if shift_class_of[sh] == k)
    for c in cycles for k in categories
}

# Balance category hours between consecutive cycles (c04 chains them: c1 ~ c2 ~ c3).
# Soft: deviation goes into the objective. Big-M relaxes the pair when the higher
# cycle is inactive (same trick as c07), so we only balance actually-used cycles.
pair_cycles = range(1, MAX_NB_CYLCEs)  # pairs (c, c+1)
d2_pos = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_pos")
d2_neg = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_neg")

BIG_M_HOURS = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())

for c in pair_cycles:
    for k in categories:
        diff = cat_hours[c, k] - cat_hours[c+1, k]
        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) <=  BIG_M_HOURS * (1 - active_cycle[c+1]),
                             name=f"Pesch2_bal_up_c{c}_k{k}")
        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) >= -BIG_M_HOURS * (1 - active_cycle[c+1]),
                             name=f"Pesch2_bal_lo_c{c}_k{k}")

$$ \sum_{c=1}^{M}{x_{c,w,d,s}}\text{ } \forall w,d,s$$

In [53]:
# additional decision variables to link shift classes to active shifts/shiftWeek/weekDay/cycle:

# binary variable indicating whether a specific class is assigned on (c,w,d), ie in cycle c, in cycleWeek w, on weekDay d
# class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k
class_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="class_assigned")

# add supporting variables for absolute differences per class between consecutive days
# diff[c,w,d,k] >= | class_assigned[c,w,d,k] - class_assigned[c,w,d+1,k] |
class_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="diff_class")


$$ \sum_{c=1}^{M}{x_{c,w,d,s}},\text{ } \forall w,d,s$$

In [54]:
# additional decision variables to link shift classes to active shifts/shiftWeek/weekDay/cycle:

# binary variable indicating whether a specific class is assigned on (c,w,d), ie in cycle c, in cycleWeek w, on weekDay d
# class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k
class_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="class_assigned")

# add supporting variables for absolute differences per class between consecutive days
# diff[c,w,d,k] >= | class_assigned[c,w,d,k] - class_assigned[c,w,d+1,k] |
class_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="diff_class")


In [ ]:
# link class_assigned to x
# for each class k, class_assigned equals the sum of x over all shifts that belong to that class
# this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
dict_shifts_by_class = {k: [sh for sh, cls in dict_shift_to_class.items() if cls == k] for k in set_classes}

for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            for k in set_classes:
                class_shifts = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
                # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
                # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
                if class_shifts:
                    modelCycle.addConstr(
                        class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in class_shifts) == 0,
                        name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
                    )
                else:
                    # if no shifts for this class (shouldn't happen by definition), force 0
                    modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
                                         name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# TASK: to be added: loop from last active week to first day of first week
for c in cycles:
    for w in range(1, MAX_CYCLE_WEEKS-1):
        for d in Weekdays:
            # determine next day index (wrap 7 -> 1)
            if d <= 6:
                d_next = d + 1
                w_next = w
            else:  # d == 7
                d_next = 1
                w_next = w+1  # use first day of next cycleWeek
            for k in set_classes:
                # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
                modelCycle.addConstr(
                    class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],
                    name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
                )
                # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
                modelCycle.addConstr(
                    class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],
                    name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
                )
# Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.

class_changes_total = gp.quicksum(class_diff[c, w, d, k] for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)

In [56]:
### seems to be duplicated cell ???
# # link class_assigned to x
# # for each class k, class_assigned equals the sum of x over all shifts that belong to that class
# # this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
# dict_shifts_by_class = {k: [sh for sh, cls in dict_shift_to_class.items() if cls == k] for k in set_classes}

# for c in cycles:
#     for w in cycleWeeks:
#         for d in Weekdays:
#             for k in set_classes:
#                 class_shifts = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
#                 # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
#                 # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
#                 if class_shifts:
#                     modelCycle.addConstr(
#                         class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in class_shifts) == 0,
#                         name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
#                     )
#                 else:
#                     # if no shifts for this class (shouldn't happen by definition), force 0
#                     modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
#                                          name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# # class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# # TASK: to be added: loop from last active week to first day of first week
# for c in cycles:
#     for w in range(1, MAX_CYCLE_WEEKS-1):
#         for d in Weekdays:
#             # determine next day index (wrap 7 -> 1)
#             if d <= 6:
#                 d_next = d + 1
#                 w_next = w
#             else:  # d == 7
#                 d_next = 1
#                 w_next = w+1  # use first day of next cycleWeek
#             for k in set_classes:
#                 # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],
#                     name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
#                 )
#                 # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],
#                     name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
#                 )
# # Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# # When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.


### objective

In [ ]:
# set objective function: minimize number of active snakes
#modelCycle.setObjective(gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks), GRB.MINIMIZE)
# minimize total number of active cycle weeks (weeks across all cycles)

#Same functionality as above, but now with weighted objectives for active weeks and fairness (Pesch1 deviation)
modelCycle.setObjective(
    W_WORKERS  * gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks)
  + W_FAIRNESS * gp.quicksum(dev_pos[c, w] + dev_neg[c, w] for c in cycles for w in cycleWeeks)
  + W_PESCH2   * gp.quicksum(d2_pos[c, k] + d2_neg[c, k] for c in pair_cycles for k in categories)
  + W_CHANGEofCLASSES * class_changes_total,
    GRB.MINIMIZE
    )


# improvements outstanding:
    # add various weighted objectives => based on user input

# Solver settings (not part of the model, only how it is solved).
# Per-week fairness made this MIP combinatorially hard: a real objective trade-off plus
# heavy cycle/week symmetry, so proving optimality is very slow while a good solution is
# found in seconds. These params make the run practical:
#   TimeLimit = 10  -> stop after 10s and return the best solution found so far
#   MIPFocus  = 1   -> prioritise finding good feasible solutions over proving the bound
#   Symmetry  = 2   -> aggressively detect and discard symmetric (identical) solutions
# The large MIP gap that remains is a weak lower bound, not a bad schedule.
modelCycle.Params.TimeLimit = 10
modelCycle.Params.MIPFocus = 1
modelCycle.Params.Symmetry = 2

#run optimizer
modelCycle.optimize()
weeks_used = sum(active_week[c, s].X for c in cycles for s in cycleWeeks)
total_dev  = sum(dev_pos[c, w].X + dev_neg[c, w].X for c in cycles for w in cycleWeeks)

# Diagnostic for the "in moeglichst vielen Turni" requirement: count how many active
# weeks hit the 40h target exactly (dev = 0). Confirms that L1 minimisation of the
# per-week deviation concentrates the unavoidable error into few weeks and leaves most
# weeks on target, so no explicit "maximise on-target weeks" constraint is needed.
active_cnt   = sum(1 for c in cycles for w in cycleWeeks if active_week[c, w].X > 0.5)
on_target    = sum(1 for c in cycles for w in cycleWeeks
                   if active_week[c, w].X > 0.5 and dev_pos[c, w].X + dev_neg[c, w].X < 1e-6)
print(f"weeks on target (dev=0): {on_target} / {active_cnt} active weeks")
print(f"active weeks: {weeks_used:.0f}, total deviation (h): {total_dev:.1f}, objective: {modelCycle.objVal:.0f}")
pesch2_imbalance = sum(d2_pos[c, k].X + d2_neg[c, k].X for c in pair_cycles for k in categories)
print(f"Pesch2 category imbalance across cycles (h): {pesch2_imbalance:.1f}")


Set parameter TimeLimit to value 10
Set parameter MIPFocus to value 1
Set parameter Symmetry to value 2
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 PRO 250 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  10
MIPFocus  1
Symmetry  2

Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 88942 rows, 16813 columns and 300472 nonzeros (Min)
Model fingerprint: 0xb7e623a1
Model has 162 linear objective coefficients
Variable types: 124 continuous, 16689 integer (16689 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [5e+01, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]

Presolve removed 85314 rows and 11950 columns
Presolve time: 0.10s
Presolved: 3628 rows, 4863 columns, 711

### results

In [58]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file => DONE
        # create a shift overview per staff member

## printing output
# if modelCycle.status == GRB.OPTIMAL:
#     print("\nminimum number of cycle weeks:", int(modelCycle.objVal))
#     abd.writeToLogs(f"successfully finished cycle plan: found an optimal solution using {int(modelCycle.objVal)} cycle weeks",FOLDER_AND_FILE_LOG)
#     for c1 in cycles:
#         if active_cycle[c1].X == 1:
#             print(f"\ncycle {c1}:")
#             abd.writeToLogs(f"\ncycle {c1}:",FOLDER_AND_FILE_LOG)
#             for w in cycleWeeks:
#                 if active_week[c1, w].X == 1:
#                     print(f"\ncycle week {w}:")
#                     abd.writeToLogs(f"\ncycle week {w}:",FOLDER_AND_FILE_LOG)
#                     for d in Weekdays:
#                         for sh in Shifts:
#                             if x[c1, w, d, sh].X == 1:
#                                 print(f"\t{DICT_WEEKDAYS_RETURN[d]}: {sh.split("%_%", 1)[0]}")
#                                 abd.writeToLogs(f"\tday {d}: {sh.split("%_%", 1)[0]}",FOLDER_AND_FILE_LOG)


# output to file
#if modelCycle.status == GRB.OPTIMAL:
#    output_string = ""
#    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:
#        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;\n")
#    for c in cycles:
#        if active_cycle[c].X > 0.5:
#            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
#                file.write(f"CYCLE: {c}:\n")
#            for w in cycleWeeks:
#                if active_week[c, w].X > 0.5:
#                    for d in Weekdays:
#                        for sh in Shifts:
#                            if x[c, w, d, sh].X  > 0.5:
#                                output_string = output_string + sh.split("%_%", 1)[0] + ";"
#                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
#                        file.write(output_string + "\n")
#                    output_string = ""


#
# Output rewritten for two reasons:
# 1. Guard changed from `status == GRB.OPTIMAL` to `SolCount > 0`. With a TimeLimit the
#    solver returns status TIME_LIMIT, not OPTIMAL, so the old guard was false and the
#    file was never updated (we were reading a stale run). SolCount > 0 writes whenever
#    any feasible solution was found.
# 2. Added three per-week columns: WorkHours (net work time of the week), Deviation
#    (signed gap from the 40h target), and OnTarget (yes/no). Values come straight from
#    the Pesch1 dev_pos/dev_neg variables, so this just reports what the model optimised.
if modelCycle.SolCount > 0:   # was == GRB.OPTIMAL; with TimeLimit status is TIME_LIMIT, so accept any found solution
    output_string = ""
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;WorkHours;Deviation;OnTarget;\n")
    for c in cycles:
        if active_cycle[c].X > 0.5:
            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                file.write(f"CYCLE: {c}:\n")
            for w in cycleWeeks:
                if active_week[c, w].X > 0.5:
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, w, d, sh].X > 0.5:
                                output_string = output_string + sh.split("%_%", 1)[0] + ";"
                    # per-week net work hours and deviation from the 40h target (from Pesch1 dev vars)
                    signed_dev = dev_pos[c, w].X - dev_neg[c, w].X
                    week_h     = AVG_WEEKLY_HOURS + signed_dev
                    on_target  = "yes" if abs(signed_dev) < 1e-6 else "no"
                    output_string += f"{week_h:.1f};{signed_dev:+.1f};{on_target};"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""



#t